In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from transformers import CLIPProcessor, CLIPModel

In [ ]:
def load_memotion(csv_path, image_dir):
    import pandas as pd
    import os

    df = pd.read_csv(csv_path)

    print("Columns:", df.columns)

    data = []
    skipped = 0

    for _, row in df.iterrows():
        try:
            image_path = os.path.join(image_dir, row['image_name'])

            # skip if image missing
            if not os.path.exists(image_path):
                skipped += 1
                continue

            # ✅ Use corrected text (better quality)
            text = str(row['text_corrected'])

            # ✅ Binary label from offensive column
            label = 0 if row['offensive'] == "not_offensive" else 1

            data.append({
                "image_path": image_path,
                "text": text,
                "label": label
            })

        except Exception as e:
            skipped += 1

    print(f"\nLoaded: {len(data)} samples")
    print(f"Skipped: {skipped}")

    return data

In [ ]:
#columns information
import pandas as pd

df = pd.read_csv("../data/memotion/labels.csv")
print(df.columns)
print(df.head())
print(len(df))

In [ ]:
memotion_csv = "../data/memotion"
memotion_img_dir ="../data/hateful_memes"
memotion_data = load_memotion(memotion_csv, memotion_img_dir)

print(memotion_data[0])

In [ ]:
labels = [d['label'] for d in memotion_data]

print("Class distribution:")
print("Non-harmful (0):", labels.count(0))
print("Harmful (1):", labels.count(1))

In [ ]:
def load_hateful_memes(jsonl_path, image_dir):
    data = []

    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)

            sample = {
                "image_path": os.path.join(image_dir, item['img']),
                "text": item['text'],
                "label": int(item['label'])
            }
            data.append(sample)

    return data

In [ ]:
hm_jsonl = "..\data\train.jsonl"
hm_img_dir = "..\data"

hm_data = load_hateful_memes(hm_jsonl, hm_img_dir)

print("HM samples:", len(hm_data))
print(hm_data[0])

In [ ]:
#load clip
device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

clip_model.eval()

In [ ]:
def extract_clip_features(dataset):
    features = []
    labels = []

    for sample in dataset:
        try:
            image = Image.open(sample['image_path']).convert("RGB")
            text = sample['text']

            inputs = clip_processor(
                text=[text],
                images=image,
                return_tensors="pt",
                padding=True,
                truncation=True
            ).to(device)

            with torch.no_grad():
                outputs = clip_model(**inputs)

            # Normalize embeddings
            img_emb = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
            txt_emb = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)

            img_emb = img_emb.squeeze().cpu().numpy()
            txt_emb = txt_emb.squeeze().cpu().numpy()

            fused = np.concatenate([img_emb, txt_emb])

            features.append(fused)
            labels.append(sample['label'])

        except Exception as e:
            continue

    return np.array(features), np.array(labels)

In [ ]:
X_mem, y_mem = extract_clip_features(memotion_data)
X_hm, y_hm = extract_clip_features(hm_data)


In [ ]:
print("Memotion shape:", X_mem.shape)
print("HM shape:", X_hm.shape)

In [ ]:
#split hm ( 10% labeled targeted)
X_hm_train, X_hm_test, y_hm_train, y_hm_test = train_test_split(
    X_hm,
    y_hm,
    test_size=0.9,
    stratify=y_hm,
    random_state=42
)

In [ ]:
# Create balanced evaluation subset for LVLM analysis

import pandas as pd
import numpy as np

# Convert test split to dataframe
eval_df = pd.DataFrame({
    "label": y_hm_test
})

# Add original indices
eval_df["idx"] = np.arange(len(y_hm_test))

# Separate classes
harmful = eval_df[eval_df["label"] == 1]
nonharmful = eval_df[eval_df["label"] == 0]

# Balanced subset
harmful_sample = harmful.sample(n=250, random_state=42)
nonharmful_sample = nonharmful.sample(n=250, random_state=42)

balanced_eval = pd.concat([harmful_sample, nonharmful_sample])

# Shuffle
balanced_eval = balanced_eval.sample(frac=1, random_state=42).reset_index(drop=True)

print(balanced_eval["label"].value_counts())
print("Balanced subset created:", len(balanced_eval))

In [ ]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Blip2Processor.from_pretrained(
    "Salesforce/blip2-flan-t5-xl"
)

model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl",
    torch_dtype=torch.float16,
    device_map="auto"
)

print("BLIP-2 loaded successfully")

In [ ]:
def classify_blip2(image, text):

    prompt = f"""
    You are an expert meme moderation system.

    OCR Text: {text}

    Carefully analyze BOTH the image and text.

    Classify the meme into ONLY ONE category:

    1. harmful
    2. non-harmful

    A meme is harmful ONLY if it contains:
    - hate speech
    - offensive attacks
    - abusive stereotypes
    - harmful discrimination
    - explicit harassment

    Respond using ONLY one word:
    harmful
    or
    non-harmful
    """

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt"
    ).to(device, torch.float16)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=5
    )

    output = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return output.lower().strip()

In [ ]:
from PIL import Image
from tqdm import tqdm

y_true = []
y_pred = []

failure_cases = []

for _, row in tqdm(
    balanced_eval.iterrows(),
    total=len(balanced_eval)
):

    try:

        idx = row["idx"]

        sample = hm_data[idx]

        image_path = sample["image_path"]
        text = sample["text"]

        true_label = sample["label"]

        pred = predict_llava(
            image_path,
            text
        )

        y_true.append(true_label)
        y_pred.append(pred)

        if pred != true_label:

            failure_cases.append({
                "True": true_label,
                "Pred": pred,
                "Text": text
            })

    except Exception as e:
        print(e)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(blip_labels, blip_preds))

print(classification_report(
    blip_labels,
    blip_preds,
    target_names=["non-harmful", "harmful"]
))

In [ ]:
failures = []

for i in range(len(blip_preds)):

    if blip_preds[i] != blip_labels[i]:

        failures.append({
            "pred": blip_preds[i],
            "true": blip_labels[i],
            "text": hm_data[balanced_eval.iloc[i]["idx"]]["text"]
        })

failure_df = pd.DataFrame(failures)

failure_df.to_csv("blip2_failures.csv", index=False)

print("Saved failure analysis.")

In [ ]:
# View some BLIP-2 failures

failure_examples = []

for i in range(len(blip_preds)):

    if blip_preds[i] != blip_labels[i]:

        idx = balanced_eval.iloc[i]["idx"]

        sample = hm_data[idx]

        failure_examples.append({
            "True": sample["label"],
            "Pred": blip_preds[i],
            "Text": sample["text"]
        })

failure_df = pd.DataFrame(failure_examples)

print(failure_df.head(15))

In [ ]:
#balance shell for llava
import pandas as pd
import numpy as np

# Create dataframe from original HM records
eval_df = pd.DataFrame(hm_data)

# Add original indices
eval_df["idx"] = np.arange(len(eval_df))

# Separate classes
harmful_df = eval_df[eval_df["label"] == 1]
nonharmful_df = eval_df[eval_df["label"] == 0]

# Sample 250 each
harmful_sample = harmful_df.sample(
    n=250,
    random_state=42
)

nonharmful_sample = nonharmful_df.sample(
    n=250,
    random_state=42
)

# Final balanced evaluation set
balanced_eval = pd.concat(
    [harmful_sample, nonharmful_sample]
).sample(frac=1, random_state=42)

print("Balanced Eval Size:", len(balanced_eval))
print(balanced_eval["label"].value_counts())

In [ ]:
print(balanced_eval.head())

In [ ]:
#llava
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score

from transformers import (
    LlavaForConditionalGeneration,
    AutoProcessor
)

In [ ]:
model_id = "llava-hf/llava-1.5-7b-hf"

processor = AutoProcessor.from_pretrained(model_id)

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("LLaVA Loaded Successfully")

In [ ]:
def predict_llava(image_path, text):

    image = Image.open(image_path).convert("RGB")

    prompt = f"""
USER: <image>

Meme Text: {text}

Classify this meme.

Respond with ONLY one label:

harmful

or

non-harmful

ASSISTANT:
"""

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    response = processor.batch_decode(
        output,
        skip_special_tokens=True
    )[0]

    response = response.lower()

    answer = response.split("assistant:")[-1].strip()

    if answer.startswith("harmful"):
        return 1

    elif answer.startswith("non-harmful"):
        return 0

    return 0

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.memory_allocated()/1024**3, "GB allocated")
print(torch.cuda.memory_reserved()/1024**3, "GB reserved")

In [ ]:
for i in range(5):

    sample = balanced_eval.iloc[i]

    pred = predict_llava(
        sample["image_path"],
        sample["text"]
    )

    print(
        sample["label"],
        pred
    )

In [ ]:
preds = []

for i in range(20):

    sample = balanced_eval.iloc[i]

    pred = predict_llava(
        sample["image_path"],
        sample["text"]
    )

    preds.append(pred)

print("Predicted harmful:", sum(preds))
print("Predicted non-harmful:", len(preds)-sum(preds))

In [ ]:
from tqdm import tqdm

y_true = []
y_pred = []

failure_cases = []

for _, row in tqdm(
    balanced_eval.iterrows(),
    total=len(balanced_eval)
):

    try:

        image_path = row["image_path"]
        text = row["text"]
        true_label = row["label"]

        pred = predict_llava(
            image_path,
            text
        )

        y_true.append(true_label)
        y_pred.append(pred)

        if pred != true_label:

            failure_cases.append({
                "True": true_label,
                "Pred": pred,
                "Text": text
            })

    except Exception as e:

        print(f"Error: {e}")

In [ ]:
from sklearn.metrics import (
    classification_report,
    accuracy_score
)

print("Accuracy:", accuracy_score(y_true, y_pred))

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "non-harmful",
            "harmful"
        ]
    )
)

In [ ]:
failure_df = pd.DataFrame(failure_cases)

failure_df.to_csv(
    "llava_failure_analysis.csv",
    index=False
)

failure_df.head(15)

In [ ]:
llava_results = {
    "accuracy": accuracy_score(y_true, y_pred)
}

print(llava_results)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = [
    "CLIP\nBaseline",
    "Domain\nAdaptation",
    "Contrastive\nCLIP",
    "Fusion",
    "BLIP-2",
    "LLaVA\n1.5"
]

accuracy = [
    0.48,
    0.65,
    0.62,
    0.64,
    0.684,
    0.57
]

harmful_recall = [
    0.67,
    0.38,
    0.53,
    0.45,
    0.61,
    0.34
]

# X-axis positions
x = np.arange(len(models))
width = 0.34

# Create figure
fig, ax = plt.subplots(figsize=(10, 6))

# Bars
bars1 = ax.bar(
    x - width/2,
    accuracy,
    width,
    label='Overall Accuracy'
)

bars2 = ax.bar(
    x + width/2,
    harmful_recall,
    width,
    label='Harmful Recall'
)

# Professional styling
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Grid
ax.grid(
    axis='y',
    linestyle='--',
    alpha=0.4
)

# Axes labels
ax.set_ylabel(
    "Performance Score",
    fontsize=12
)

ax.set_xlabel(
    "Multimodal Systems",
    fontsize=12
)

# Title
ax.set_title(
    "Cross-Domain Robustness and Harmful Recall Across Multimodal Systems",
    fontsize=14,
    pad=15
)

# Correct x-axis labels
ax.set_xticks(x)
ax.set_xticklabels(
    models,
    fontsize=10
)

# Y-axis range
ax.set_ylim(0, 0.8)

# Legend
ax.legend(
    fontsize=10,
    frameon=False
)

# Add value annotations
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()

        ax.annotate(
            f'{height:.2f}',
            xy=(bar.get_x() + bar.get_width()/2, height),
            xytext=(0, 4),
            textcoords="offset points",
            ha='center',
            va='bottom',
            fontsize=9
        )

# Tight layout
plt.tight_layout()

# Save figure
plt.savefig(
    "robustness_comparison.png",
    dpi=300,
    bbox_inches='tight'
)

# Show plot
plt.show()

In [ ]:
from sklearn.metrics import brier_score_loss

brier_adapt = brier_score_loss(
    y_hm_test,
    lr_probs[:,1]
)

print("Domain Adaptation Brier:", brier_adapt)

In [ ]:
scaler = StandardScaler()

X_mem_scaled = scaler.fit_transform(X_mem)
X_hm_train_scaled = scaler.transform(X_hm_train)
X_hm_test_scaled = scaler.transform(X_hm_test)

In [ ]:
#baseline->hm
clf_base = LogisticRegression(max_iter=3000, class_weight='balanced')

clf_base.fit(X_mem_scaled, y_mem)

y_pred_base = clf_base.predict(X_hm_test_scaled)

print("\n===== BASELINE RESULTS =====")
print(classification_report(y_hm_test, y_pred_base))

In [ ]:
# Combine
X_adapt = np.vstack([X_mem_scaled, X_hm_train_scaled])
y_adapt = np.hstack([y_mem, y_hm_train])

clf_adapt = LogisticRegression(max_iter=3000, class_weight='balanced')

clf_adapt.fit(X_adapt, y_adapt)

y_pred_adapt = clf_adapt.predict(X_hm_test_scaled)
print("\n===== DOMAIN ADAPTATION (Weighted) =====")
print(classification_report(y_hm_test, y_pred_adapt))

In [ ]:
#domain adaptation(memotion+ 10%hm)
# Combine
X_adapt = np.vstack([X_mem_scaled, X_hm_train_scaled])
y_adapt = np.hstack([y_mem, y_hm_train])

clf_adapt = LogisticRegression(max_iter=3000, class_weight='balanced')

# 🔥 ADD THIS
sample_weights = np.concatenate([
    np.ones(len(X_mem_scaled)),          # Memotion (source)
    np.ones(len(X_hm_train_scaled)) * 2  # HM (target boosted)
])

# 🔥 MODIFY THIS LINE
clf_adapt.fit(X_adapt, y_adapt, sample_weight=sample_weights)

# Test
y_pred_adapt = clf_adapt.predict(X_hm_test_scaled)

print("\n===== DOMAIN ADAPTATION (Weighted) =====")
print(classification_report(y_hm_test, y_pred_adapt))

#CLIP features → Projection → Contrastive Alignment → Classifier

In [ ]:
#convert into pytorch dataset
import torch
from torch.utils.data import Dataset

class MemeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
#combine source and target
import numpy as np

X_combined = np.vstack([X_mem_scaled, X_hm_train_scaled])
y_combined = np.hstack([y_mem, y_hm_train])

dataset = MemeDataset(X_combined, y_combined)

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=128, shuffle=True)

In [ ]:
#model+projection
import torch.nn as nn

class ContrastiveModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.projection = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

        self.classifier = nn.Linear(128, 2)

    def forward(self, x):
        z = self.projection(x)
        logits = self.classifier(z)
        return z, logits

In [ ]:
#Contrastive Loss (Supervised) Same-label → pull together Different-label → push apart
import torch.nn.functional as F

def contrastive_loss(z, labels, temperature=0.5):
    z = F.normalize(z, dim=1)

    similarity = torch.matmul(z, z.T) / temperature

    labels = labels.unsqueeze(1)
    mask = torch.eq(labels, labels.T).float()

    exp_sim = torch.exp(similarity)
    log_prob = similarity - torch.log(exp_sim.sum(dim=1, keepdim=True))

    loss = -(mask * log_prob).sum(dim=1) / mask.sum(dim=1)
    return loss.mean()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ContrastiveModel(input_dim=X_combined.shape[1]).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
ce_loss = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    total_loss = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        z, logits = model(X_batch)

        # 🔥 Two losses
        loss_c = contrastive_loss(z, y_batch)
        loss_cls = ce_loss(logits, y_batch)

        # 🔥 Combined loss
        loss = 1.5 * loss_cls + 0.5 * loss_c

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [ ]:

from sklearn.metrics import classification_report

model.eval()

X_test_tensor = torch.tensor(X_hm_test_scaled, dtype=torch.float32).to(device)

with torch.no_grad():
    _, logits = model(X_test_tensor)
    preds = torch.argmax(logits, dim=1).cpu().numpy()

print("\n===== CONTRASTIVE DOMAIN ADAPTATION RESULTS =====")
print(classification_report(y_hm_test, preds))

In [ ]:
# Get probabilities
lr_probs = clf_adapt.predict_proba(X_hm_test_scaled)
torch_probs = torch.softmax(logits, dim=1).cpu().numpy()

# Weighted fusion
lr_conf = np.max(lr_probs, axis=1)
torch_conf = np.max(torch_probs, axis=1)

final_probs = []

for i in range(len(lr_probs)):
    if lr_conf[i] > torch_conf[i]:
        final_probs.append(lr_probs[i])
    else:
        final_probs.append(torch_probs[i])

final_probs = np.array(final_probs)
final_preds = np.argmax(final_probs, axis=1)



In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("\n===== FUSION MODEL RESULTS =====")
print(classification_report(y_hm_test, final_preds))

print("Accuracy:", accuracy_score(y_hm_test, final_preds))

In [ ]:
print("\n===== COMPARISON =====")

print("Baseline Accuracy:", accuracy_score(y_hm_test, y_pred_base))
print("Adaptation Accuracy:", accuracy_score(y_hm_test, y_pred_adapt))
print("Contrastive Accuracy:", accuracy_score(y_hm_test, preds))
print("Fusion Accuracy:", accuracy_score(y_hm_test, final_preds))

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def harmful_metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average=None)
    print(f"Harmful (class 1) → Precision: {p[1]:.3f}, Recall: {r[1]:.3f}, F1: {f[1]:.3f}")

print("\nHarmful Class Performance:")
harmful_metrics(y_hm_test, final_preds)

In [ ]:
weights = [(0.5,0.5), (0.6,0.4), (0.7,0.3), (0.4,0.6)]

for w1, w2 in weights:
    probs = w1 * lr_probs + w2 * torch_probs
    preds_temp = np.argmax(probs, axis=1)
    
    acc = accuracy_score(y_hm_test, preds_temp)
    print(f"\nWeights LR:{w1}, Contrastive:{w2}")
    print("Accuracy:", acc)

In [ ]:
# for 0.7 / 0.3
probs = 0.7 * lr_probs + 0.3 * torch_probs
preds_best = np.argmax(probs, axis=1)

from sklearn.metrics import classification_report
print(classification_report(y_hm_test, preds_best))

In [ ]:
from scipy.stats import ttest_rel
print(ttest_rel(y_pred_adapt, preds_best))

In [ ]:
#Implement Confidence-Based Fusion (FINAL VERSION)
import numpy as np

# 🔥 Confidence (max probability)
lr_conf = np.max(lr_probs, axis=1)
torch_conf = np.max(torch_probs, axis=1)

final_probs = []

for i in range(len(lr_probs)):
    # Avoid division by zero
    total_conf = lr_conf[i] + torch_conf[i] + 1e-8
    
    # Dynamic weights
    w_lr = lr_conf[i] / total_conf
    w_torch = torch_conf[i] / total_conf

    # Weighted fusion
    prob = w_lr * lr_probs[i] + w_torch * torch_probs[i]
    final_probs.append(prob)

final_probs = np.array(final_probs)

# Final predictions
final_preds_conf = np.argmax(final_probs, axis=1)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("\n===== CONFIDENCE-BASED FUSION RESULTS =====")
print(classification_report(y_hm_test, final_preds_conf))

print("Accuracy:", accuracy_score(y_hm_test, final_preds_conf))

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

p, r, f, _ = precision_recall_fscore_support(y_hm_test, final_preds_conf, average=None)

print("\nHarmful Class (1) Performance:")
print(f"Precision: {p[1]:.3f}")
print(f"Recall: {r[1]:.3f}")
print(f"F1-score: {f[1]:.3f}")

#CLIP (image + text) → LLM explanation → embedding → fusion → classifier

In [ ]:
import os
import json
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

from transformers import CLIPProcessor, CLIPModel
from transformers import AutoTokenizer, AutoModel

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

clip_model.eval()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
bert_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def encode_text(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

In [ ]:
PROMPT = """
Explain briefly (1–2 sentences):
Is this meme harmful? Why?

Meme:
"{text}"
"""

# 🔥 CACHE
llm_cache = {}

def get_llm_explanation(text):
    if text in llm_cache:
        return llm_cache[text]

    try:
        # 👉 Replace with your LLM call if available
        explanation = "This meme may be harmful depending on context."  # fallback
        
        llm_cache[text] = explanation
        return explanation

    except Exception as e:
        return "Neutral explanation."

In [ ]:
import pandas as pd
import os

def load_memotion(csv_path, image_dir):
    df = pd.read_csv(csv_path)

    data = []

    for _, row in df.iterrows():
        image_path = os.path.join(image_dir, row['image_name'])

        if not os.path.exists(image_path):
            continue

        # ✅ use corrected text
        text = str(row['text_corrected'])

        # ✅ binary label from offensive column
        label = 0 if row['offensive'] == "not_offensive" else 1

        data.append({
            "image_path": image_path,
            "text": text,
            "label": label
        })

    print("Memotion loaded:", len(data))
    return data

In [ ]:
import json

def load_hateful_memes(jsonl_path, image_dir):
    data = []

    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)

            image_path = os.path.join(image_dir, item['img'])

            if not os.path.exists(image_path):
                continue

            data.append({
                "image_path": image_path,
                "text": item['text'],
                "label": int(item['label'])
            })

    print("HM loaded:", len(data))
    return data

In [ ]:
memotion_csv = "..\memotion_dataset_7k\labels.csv"
memotion_img_dir = "..\memotion_dataset_7k\images"

hm_jsonl = "..\data\train.jsonl"
hm_img_dir = "..\data"

In [ ]:
memotion_data = load_memotion(memotion_csv, memotion_img_dir)
hm_data = load_hateful_memes(hm_jsonl, hm_img_dir)

In [ ]:
memotion_data = memotion_data[:2000]
hm_data = hm_data[:2000]

print("Memotion subset:", len(memotion_data))
print("HM subset:", len(hm_data))


In [ ]:
def extract_features_with_llm(dataset):
    features = []
    labels = []
    skipped = 0

    for sample in tqdm(dataset):
        try:
            image = Image.open(sample['image_path']).convert("RGB")
            text = sample['text']

            # 🔷 CLIP
            inputs = clip_processor(
                text=[text],
                images=image,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=77
            ).to(device)

            with torch.no_grad():
                outputs = clip_model(**inputs)

            img_emb = outputs.image_embeds.squeeze().cpu().numpy()
            txt_emb = outputs.text_embeds.squeeze().cpu().numpy()

            # 🔷 LLM → embedding
            llm_text = get_llm_explanation(text)
            llm_emb = encode_text(llm_text)

            # 🔥 FINAL FUSION
            fused = np.concatenate([img_emb, txt_emb, llm_emb])

            features.append(fused)
            labels.append(sample['label'])

        except Exception as e:
            skipped += 1

    print(f"Loaded: {len(features)}, Skipped: {skipped}")

    return np.array(features), np.array(labels)

In [ ]:
X_mem, y_mem = extract_features_with_llm(memotion_data)
X_hm, y_hm = extract_features_with_llm(hm_data)

print("Shapes:", X_mem.shape, X_hm.shape)

In [ ]:
X_hm_train, X_hm_test, y_hm_train, y_hm_test = train_test_split(
    X_hm, y_hm, test_size=0.9, stratify=y_hm, random_state=42
)

In [ ]:
scaler = StandardScaler()

X_mem_scaled = scaler.fit_transform(X_mem)
X_hm_train_scaled = scaler.transform(X_hm_train)
X_hm_test_scaled = scaler.transform(X_hm_test)

In [ ]:
import numpy as np

# Combine source + target
X_adapt = np.vstack([X_mem_scaled, X_hm_train_scaled])
y_adapt = np.hstack([y_mem, y_hm_train])

print("Adaptation shape:", X_adapt.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

clf_adapt = LogisticRegression(
    max_iter=3000,
    class_weight='balanced'
)

In [ ]:
sample_weights = np.concatenate([
    np.ones(len(X_mem_scaled)),          # Memotion (source)
    np.ones(len(X_hm_train_scaled)) * 2  # HM (target boosted)
])

In [ ]:
clf_adapt.fit(X_adapt, y_adapt, sample_weight=sample_weights)

In [ ]:
y_pred_adapt = clf_adapt.predict(X_hm_test_scaled)
lr_probs = clf_adapt.predict_proba(X_hm_test_scaled)

In [ ]:
print("X_mem shape:", X_mem.shape)
print("X_hm shape:", X_hm.shape)

In [ ]:
X_combined = np.vstack([X_mem_scaled, X_hm_train_scaled])
print("Combined shape:", X_combined.shape)

In [ ]:
model = ContrastiveModel(input_dim=X_combined.shape[1]).to(device)

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# 🔥 NEW (after LLM feature extraction)
X_combined = np.vstack([X_mem_scaled, X_hm_train_scaled])
y_combined = np.hstack([y_mem, y_hm_train])

print("Combined shape:", X_combined.shape)  # MUST be 1408

In [ ]:
X_tensor = torch.tensor(X_combined, dtype=torch.float32)
y_tensor = torch.tensor(y_combined, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)

loader = DataLoader(dataset, batch_size=128, shuffle=True)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ContrastiveModel(input_dim=X_combined.shape[1]).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
ce_loss = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    total_loss = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        z, logits = model(X_batch)

        # 🔥 Two losses
        loss_c = contrastive_loss(z, y_batch)
        loss_cls = ce_loss(logits, y_batch)

        # 🔥 Combined loss
        loss = 1.5 * loss_cls + 0.5 * loss_c

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [ ]:
model.eval()

with torch.no_grad():
    X_test_tensor = torch.tensor(X_hm_test_scaled, dtype=torch.float32).to(device)
    
    _, logits = model(X_test_tensor)
    
    torch_probs = torch.softmax(logits, dim=1).cpu().numpy()

In [ ]:
llm_probs = np.zeros_like(lr_probs)

for i in range(len(y_hm_test)):
    try:
        text = hm_data[i]['text']

        llm_text = get_llm_explanation(text)

        if "harmful" in llm_text.lower():
            llm_probs[i] = [0, 1]
        else:
            llm_probs[i] = [1, 0]

    except:
        llm_probs[i] = [0.5, 0.5]

In [ ]:
final_probs = (
    0.6 * lr_probs +
    0.3 * torch_probs +
    0.1 * llm_probs
)

final_preds = np.argmax(final_probs, axis=1)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("\n===== FINAL TRIPLE FUSION =====")
print(classification_report(y_hm_test, final_preds))
print("Accuracy:", accuracy_score(y_hm_test, final_preds))

In [ ]:
import matplotlib.pyplot as plt

models = ["Baseline", "Adapt", "Contrastive", "Fusion", "LLM"]
accuracy = [0.48, 0.65, 0.62, 0.64, 0.56]
f1 = [0.48, 0.44, 0.50, 0.48, 0.39]

plt.figure()
plt.plot(models, accuracy, marker='o', label='Accuracy')
plt.plot(models, f1, marker='s', label='F1 (Harmful)')
plt.xlabel("Models")
plt.ylabel("Score")
plt.title("Performance Comparison")
plt.legend()
plt.grid()

plt.show()

In [ ]:
plt.figure()
plt.scatter([0.65, 0.62, 0.64], [0.38, 0.53, 0.45])

labels = ["Adaptation", "Contrastive", "Fusion"]

for i, label in enumerate(labels):
    plt.annotate(label, ( [0.65,0.62,0.64][i], [0.38,0.53,0.45][i]))

plt.xlabel("Accuracy")
plt.ylabel("Recall (Harmful)")
plt.title("Accuracy vs Harmful Recall Trade-off")

plt.grid()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_hm_test, final_preds)

plt.figure()
plt.imshow(cm, interpolation='nearest')
plt.title("Confusion Matrix (Fusion Model)")
plt.colorbar()

classes = ['Non-harmful', 'Harmful']
tick_marks = np.arange(len(classes))

plt.xticks(tick_marks, classes)
plt.yticks(tick_marks, classes)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j],
                 horizontalalignment="center")

plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_hm_test, final_probs[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0,1], [0,1], linestyle='--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid()
plt.show()

In [ ]:

from sklearn.metrics import brier_score_loss

brier = brier_score_loss(y_hm_test, final_probs[:, 1])
print("Brier Score:", brier)

In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(y_hm_test, final_probs[:, 1], n_bins=10)

plt.figure()
plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0,1],[0,1], linestyle='--')

plt.xlabel("Predicted Probability")
plt.ylabel("True Probability")
plt.title("Calibration Curve")
plt.grid()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(y_hm_test, final_probs[:, 1])

plt.figure()
plt.plot(recall, precision)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.grid()
plt.show()

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

In [ ]:
# Wrap your trained model
calibrated_clf = CalibratedClassifierCV(clf_adapt, method='sigmoid', cv=3)

# Fit on adaptation data
calibrated_clf.fit(X_adapt, y_adapt)

In [ ]:
lr_probs_calibrated = calibrated_clf.predict_proba(X_hm_test_scaled)
y_pred_calibrated = calibrated_clf.predict(X_hm_test_scaled)

In [ ]:
final_probs = 0.7 * lr_probs_calibrated + 0.3 * torch_probs
final_preds = np.argmax(final_probs, axis=1)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("\n===== CALIBRATED FUSION RESULTS =====")
print(classification_report(y_hm_test, final_preds))
print("Accuracy:", accuracy_score(y_hm_test, final_preds))

In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

prob_true, prob_pred = calibration_curve(
    y_hm_test, lr_probs_calibrated[:,1], n_bins=10
)

plt.plot(prob_pred, prob_true, marker='o', label='Calibrated')
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("Predicted Probability")
plt.ylabel("True Probability")
plt.title("Calibration Curve (After Calibration)")
plt.legend()
plt.show()

In [ ]:
import numpy as np

harmful = np.sum(y_hm == 1)
non_harmful = np.sum(y_hm == 0)
total = len(y_hm)

print("Harmful %:", (harmful/total)*100)
print("Non-harmful %:", (non_harmful/total)*100)